# Machine Learning Project — Weeks 1 to 6
**Dataset:** Loan Default Prediction (Banking) — Loan_default.csv
---
**Tasks Covered:**
1. **Week 1 & 2:** Data Exploration, Cleaning & Preprocessing (EDA, One-Hot Encoding, StandardScaler)
2. **Task 3 (Week 3):** Model Creation (Sklearn LogisticRegression) + **Gradient Descent from Scratch (Part B)**
3. **Task 4 (Week 4):** Prediction UI (React + Vite Web Terminal — Completed & Preserved)
4. **Week 5:** Advanced Model Training (Logistic Regression, Random Forest, AdaBoost, Gradient Boosting), 5-Fold Cross-Validation, Fit Assessment & Hyperparameter Tuning (GridSearchCV)
5. **Week 6:** Visualization of Performance Metrics & Graphs (Confusion Matrix, ROC Curves, PR Curves, Feature Importances, CV Distribution Boxplot, Model Benchmark Chart)


In [1]:
# ==============================================================================
# ML LAB ASSIGNMENT — WEEKS 1 TO 6
# Dataset: Loan Default Prediction (Banking) — Loan_default.csv
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
    precision_recall_curve, average_precision_score
)


In [2]:
# --- ---------------------------------------------------------------------------
# WEEK 1 & 2: DATA EXPLORATION, CLEANING & PREPROCESSING (Existing Task Code)


In [3]:
# --- ---------------------------------------------------------------------------
df = pd.read_csv("Loan_default.csv")

print("Dataset Head:")
print(df.head(10))

print("\nDataset Shape:", df.shape[0], "rows,", df.shape[1], "columns")
print("\nData Types:")
print(df.dtypes)
print("\nDescriptive Statistics:")
print(df.describe())
print("\nUnique Values Count:")
print(df.nunique())
print("\nDefault Class Distribution:")
print(df["Default"].value_counts())

print("\nAverage Income:", df["Income"].mean())
print("Min Loan Amount:", df["LoanAmount"].min(), "| Max Loan Amount:", df["LoanAmount"].max())
print("Average Credit Score:", df["CreditScore"].mean())
print("Education Breakdown:\n", df[df["Education"].isin(["Master's", "PhD"])]["Education"].value_counts())
print("Has Mortgage Count:", df[df["HasMortgage"] == "Yes"].shape[0])
print("Min Age:", df["Age"].min(), "| Max Age:", df["Age"].max())
print("Top 5 Highest Incomes:\n", df.sort_values("Income", ascending=False).head(5))
print("High Credit Score (>700):\n", df[df["CreditScore"] > 700].head())
print("Employment Type Counts:\n", df['EmploymentType'].value_counts())
print("Marital Status Counts:\n", df["MaritalStatus"].value_counts())
print("Mean Loan Amount by Employment Type:\n", df.groupby("EmploymentType")["LoanAmount"].mean())
print("Interest Rate Std:", np.std(df["InterestRate"]))
print("Mean Credit Score & Default by Loan Purpose:\n", df.groupby("LoanPurpose")[["CreditScore", "Default"]].mean())

print("Missing Values Check:", df.isnull().sum().to_dict())
print("Duplicated Rows Check:", df.duplicated().sum())

# One-Hot Encoding
dfd = pd.get_dummies(df, columns=[
    'Education', 'EmploymentType', 'MaritalStatus',
    'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner'
], drop_first=True)

cols = ['Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Age']
scaler = StandardScaler()
dfd[cols] = scaler.fit_transform(dfd[cols])


Dataset Head:
       LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   
5  A9S62RQ7US   25   90298       90448          720              18   
6  H8GXPAOS71   38  111188      177025          429              80   
7  0HGZQKJ36W   56  126802      155511          531              67   
8  1R0N3LGNRJ   36   42053       92357          827              83   
9  CM9L1GTT2P   40  132784      228510          480             114   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0               4         15.23        36      0.44   Bachelor's   
1               1          4.81        60      0.68     Master's   


In [ ]:
# --- ---------------------------------------------------------------------------
# TASK 3: MODEL CREATION & GRADIENT DESCENT IMPLEMENTATION


In [4]:
# --- ---------------------------------------------------------------------------

# Part A: Model Creation using Scikit-Learn (Existing Code)
X = dfd.drop(['Default', 'LoanID'], axis=1)
y = dfd['Default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nTask 3 Part A (Scikit-Learn Logistic Regression) Predictions Sample:", y_pred[:10])

# Part B: Gradient Descent Implementation from Scratch (Added as required)
print("\n--- TASK 3 PART B: GRADIENT DESCENT IMPLEMENTATION FROM SCRATCH ---")

X_gd = X_train.values.astype(np.float64)
y_gd = y_train.values.astype(np.float64).reshape(-1, 1)

n_samples, n_features = X_gd.shape
weights = np.zeros((n_features, 1))
bias = 0.0

lr = 0.01
epochs = 500

def sigmoid(z):
    z_clipped = np.clip(z, -250.0, 250.0)
    return 1.0 / (1.0 + np.exp(-z_clipped))

print("Running Gradient Descent for 500 epochs...")
for epoch in range(epochs):
    y_pred_prob = sigmoid(np.dot(X_gd, weights) + bias)
    error = y_pred_prob - y_gd
    
    # Binary Cross-Entropy Loss
    loss = -np.mean(y_gd * np.log(y_pred_prob + 1e-15) + (1 - y_gd) * np.log(1 - y_pred_prob + 1e-15))
    
    # Gradients w.r.t weights and bias
    dw = (1.0 / n_samples) * np.dot(X_gd.T, error)
    db = (1.0 / n_samples) * np.sum(error)
    
    # Update parameters
    weights -= lr * dw
    bias -= lr * db

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:3d} | Log-Loss (BCE): {loss:.4f}")

print("\nGradient Descent Parameters Comparison:")
print("Sklearn Coefficients (first 3):", model.coef_[0][:3])
print("Scratch Weights (first 3):", weights.ravel()[:3])
print("Sklearn Intercept:", model.intercept_[0])
print("Scratch Intercept:", bias)



Task 3 Part A (Scikit-Learn Logistic Regression) Predictions Sample: [0 0 0 1 0 0 1 0 1 1]

--- TASK 3 PART B: GRADIENT DESCENT IMPLEMENTATION FROM SCRATCH ---
Running Gradient Descent for 500 epochs...
Epoch   0 | Log-Loss (BCE): 0.6931
Epoch 100 | Log-Loss (BCE): 0.4738
Epoch 200 | Log-Loss (BCE): 0.4014
Epoch 300 | Log-Loss (BCE): 0.3711
Epoch 400 | Log-Loss (BCE): 0.3558
Epoch 499 | Log-Loss (BCE): 0.3470

Gradient Descent Parameters Comparison:
Sklearn Coefficients (first 3): [-0.58277528 -0.31383953  0.28739621]
Scratch Weights (first 3): [-0.18528023 -0.11037838  0.0960816 ]
Sklearn Intercept: -0.06570379365185665
Scratch Intercept: -0.6945112618142412


In [5]:
# --- ---------------------------------------------------------------------------
# WEEK 5: ADVANCED MODEL TRAINING, EVALUATION, CROSS-VALIDATION & TUNING


In [6]:
# --- ---------------------------------------------------------------------------
print("\n=================================================================")
print("WEEK 5: ADVANCED MODEL TRAINING, EVALUATION, CV & TUNING")
print("=================================================================")

X_train_sub = X_train.iloc[:25000]
y_train_sub = y_train.iloc[:25000]
X_test_sub = X_test.iloc[:5000]
y_test_sub = y_test.iloc[:5000]

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=50, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, max_depth=5, random_state=42)
}

comparison_rows = []
cv_results = {}
trained_models = {}

for name, clf in models.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train_sub, y_train_sub)
    trained_models[name] = clf
    
    train_pred = clf.predict(X_train_sub)
    test_pred = clf.predict(X_test_sub)
    
    train_acc = accuracy_score(y_train_sub, train_pred)
    test_acc = accuracy_score(y_test_sub, test_pred)
    prec = precision_score(y_test_sub, test_pred, zero_division=0)
    rec = recall_score(y_test_sub, test_pred, zero_division=0)
    f1 = f1_score(y_test_sub, test_pred, zero_division=0)
    
    # 5-Fold Cross Validation
    cv_scores = cross_val_score(clf, X_train_sub, y_train_sub, cv=5, scoring='accuracy', n_jobs=-1)
    cv_results[name] = cv_scores
    
    # Overfitting / Underfitting check
    diff = train_acc - test_acc
    if diff > 0.10:
        fit_assessment = "Overfitting"
    elif train_acc < 0.60 and test_acc < 0.60:
        fit_assessment = "Underfitting"
    else:
        fit_assessment = "Good Fit"
        
    comparison_rows.append({
        "Model": name,
        "Train Acc": round(train_acc, 4),
        "Test Acc": round(test_acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "5-Fold CV Mean": round(cv_scores.mean(), 4),
        "5-Fold CV Std": round(cv_scores.std(), 4),
        "Fit Assessment": fit_assessment
    })

comparison_df = pd.DataFrame(comparison_rows)
print("\n--- MODEL COMPARISON TABLE (TASK 5 CHECKLIST) ---")
print(comparison_df.to_string(index=False))

# Hyperparameter Tuning using GridSearchCV on Random Forest
print("\n--- HYPERPARAMETER TUNING (GridSearchCV on Random Forest) ---")
param_grid = {
    'n_estimators': [30, 60],
    'max_depth': [8, 12]
}
grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid, cv=3, scoring='f1', n_jobs=-1
)
grid_search.fit(X_train_sub, y_train_sub)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1-Score:", round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test_sub)
print("Tuned Model Test Accuracy:", round(accuracy_score(y_test_sub, best_pred), 4))
print("Tuned Model Test F1-Score:", round(f1_score(y_test_sub, best_pred), 4))



WEEK 5: ADVANCED MODEL TRAINING, EVALUATION, CV & TUNING

Training Logistic Regression...

Training Random Forest...

Training AdaBoost...

Training Gradient Boosting...

--- MODEL COMPARISON TABLE (TASK 5 CHECKLIST) ---
              Model  Train Acc  Test Acc  Precision  Recall  F1-Score  5-Fold CV Mean  5-Fold CV Std Fit Assessment
Logistic Regression     0.6758    0.6724     0.2104  0.6807    0.3215          0.6749         0.0047       Good Fit
      Random Forest     0.8827    0.8162     0.2843  0.4035    0.3336          0.8220         0.0080       Good Fit
           AdaBoost     0.8842    0.8874     0.5897  0.0404    0.0755          0.8839         0.0004       Good Fit
  Gradient Boosting     0.8936    0.8886     0.6032  0.0667    0.1201          0.8833         0.0006       Good Fit

--- HYPERPARAMETER TUNING (GridSearchCV on Random Forest) ---
Best Parameters: {'max_depth': 8, 'n_estimators': 60}
Best CV F1-Score: 0.3431
Tuned Model Test Accuracy: 0.7604
Tuned Model Test F1-Sc

In [7]:
# --- ---------------------------------------------------------------------------
# WEEK 6: VISUALIZATION OF METRICS AND GRAPHS


In [8]:
# --- ---------------------------------------------------------------------------
print("\n=================================================================")
print("WEEK 6: METRICS AND PERFORMANCE GRAPH VISUALIZATIONS")
print("=================================================================")

# 1. Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_sub, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Default', 'Default'], yticklabels=['Non-Default', 'Default'])
plt.title('Confusion Matrix — Tuned Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.close()
print("Saved confusion_matrix.png")

# 2. ROC Curve Plot
plt.figure(figsize=(8, 6))
for name, clf in trained_models.items():
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_test_sub)[:, 1]
        fpr, tpr, _ = roc_curve(y_test_sub, y_prob)
        roc_auc_val = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.title('ROC Curves Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png')
plt.close()
print("Saved roc_curves.png")

# 3. Precision-Recall Curve Plot
plt.figure(figsize=(8, 6))
for name, clf in trained_models.items():
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_test_sub)[:, 1]
        prec_curve, rec_curve, _ = precision_recall_curve(y_test_sub, y_prob)
        ap_val = average_precision_score(y_test_sub, y_prob)
        plt.plot(rec_curve, prec_curve, label=f'{name} (AP = {ap_val:.3f})')

plt.title('Precision-Recall Curves')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_curves.png')
plt.close()
print("Saved precision_recall_curves.png")

# 4. Feature Importance Bar Chart
if hasattr(best_model, 'feature_importances_'):
    plt.figure(figsize=(9, 6))
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:10]
    top_features = [X_train_sub.columns[i] for i in indices]
    top_importances = importances[indices]
    
    sns.barplot(x=top_importances, y=top_features, palette='viridis')
    plt.title('Top 10 Feature Importances — Tuned Random Forest')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('feature_importances.png')
    plt.close()
    print("Saved feature_importances.png")

# 5. Model Metrics Comparison Bar Chart
plt.figure(figsize=(10, 5))
metrics_df = comparison_df.melt(id_vars=['Model'], value_vars=['Train Acc', 'Test Acc', 'Precision', 'Recall', 'F1-Score'], var_name='Metric', value_name='Score')
sns.barplot(data=metrics_df, x='Model', y='Score', hue='Metric', palette='Set2')
plt.title('Model Performance Metrics Comparison')
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison_metrics.png')
plt.close()
print("Saved model_comparison_metrics.png")

# 6. Cross-Validation Score Distribution Boxplot
plt.figure(figsize=(8, 5))
cv_df = pd.DataFrame(cv_results)
sns.boxplot(data=cv_df, palette='pastel')
plt.title('5-Fold Cross-Validation Accuracy Distribution')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.savefig('cv_scores_boxplot.png')
plt.close()
print("Saved cv_scores_boxplot.png")

print("\n=================================================================")
print("WEEKS 1 TO 6 ASSIGNMENT EXECUTION COMPLETED SUCCESSFULLY!")
print("=================================================================")



WEEK 6: METRICS AND PERFORMANCE GRAPH VISUALIZATIONS
Saved confusion_matrix.png
Saved roc_curves.png
Saved precision_recall_curves.png


C:\Users\MEET BAROT\AppData\Local\Temp\ipykernel_10360\803284487.py:65: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_importances, y=top_features, palette='viridis')


Saved feature_importances.png
Saved model_comparison_metrics.png
Saved cv_scores_boxplot.png

WEEKS 1 TO 6 ASSIGNMENT EXECUTION COMPLETED SUCCESSFULLY!
